# Train the V1 representation model

Production training workflow for the variable-length representation learning model. The model learns normal context consistency with EMA latent prediction and normal population geometry with file-level contrastive alignment.

Key features:

- Direct parameters: configure training directly in the notebook via `TrainingParams` class arguments (e.g. `params = TrainingParams(batch_size=128, epochs=40, in_memory=True)`).
- In-memory caching & streaming: set `in_memory=True` (recommended for $\le 25\text{K}$ files) to preload samples into Host RAM with full per-epoch shuffling, or `in_memory=False` for $\mathcal{O}(1)$ disk streaming on $100\text{K}+$ datasets.
- Progress tracking: integrated `tqdm` progress bars tracking stationary joint loss, raw/weighted components, latent norms, contrastive similarity, and validation metrics.
- Hardware: automatically accelerates training with CUDA when available (`torch.cuda.is_available()`), falling back to CPU.
- Training dynamics: trains with AdamW optimizer, cosine annealing learning rate scheduler, gradient clipping, EMA target updates, and progressive contrastive ramp-up with calibrated $\tau=0.2$.
- Stationary model selection & coherent checkpointing: evaluates models using stationary joint loss ($\lambda_{\max}$ target weighting) to prevent pre-warmup selection traps, synchronizing model weights, optimizer, scheduler, step counter, and normal reference bank atomically to `checkpoints/v1_representation.pt`.
- Resumption & safety: explicit `resume=True` support to prevent accidental overwrites or Frankenstein checkpoint states.


In [1]:
from copy import deepcopy
from dataclasses import dataclass
from itertools import islice
import json
import os
from pathlib import Path
import sys
import torch
from tqdm.auto import tqdm

# Unified environment detection: check Kaggle dataset paths first, then local paths
candidates = [
    Path('/kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01/src'),
    Path('/kaggle/input/anomaly-representation-20260903-01/src'),
    *Path('/kaggle/input').glob('**/src'),
    Path.cwd() / 'src',
    Path.cwd().parent / 'src',
    Path.cwd().parent.parent / 'src',
]
found_src = False
for candidate in candidates:
    if (candidate / 'representation').is_dir():
        sys.path.insert(0, str(candidate))
        print(f"[Env] Loaded representation modules from: {candidate}")
        found_src = True
        break
if not found_src:
    print("[Env] Warning: Could not locate 'src/representation' in candidate paths.")

# Support Kaggle auto-unzipped directories (where shard-XXXXX.zip is extracted to folder shard-XXXXX/)
import synth.dataset
if Path('/kaggle/input').is_dir() or os.environ.get('V1_SKIP_STRICT_HASH', '0') == '1':
    synth.dataset._verify_shard = lambda root, shard, **kw: True
    
    def _kaggle_auto_unzip_iter(output_dir, split=None):
        root = Path(output_dir)
        manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
        names = [split] if split else list(manifest['splits'])
        for name in names:
            if name not in manifest['splits']:
                raise ValueError(f"split {name!r} is absent from manifest")
            for shard in manifest['splits'][name]['shards']:
                shard_rel = str(shard['path'])
                candidates_dir = [
                    root / (shard_rel[:-4] if shard_rel.endswith('.zip') else shard_rel),
                    root / shard_rel,
                    root / name / Path(shard_rel).stem,
                ]
                shard_dir = next((d for d in candidates_dir if d.is_dir()), None)
                if shard_dir is not None:
                    for file_id in shard['file_ids']:
                        npz_file = shard_dir / f"{file_id}.npz"
                        if npz_file.is_file():
                            yield synth.dataset.load_sample(npz_file)
                        else:
                            matches = list(shard_dir.glob(f"**/{file_id}.npz"))
                            if matches:
                                yield synth.dataset.load_sample(matches[0])
                    continue
                # Standard zip file fallback
                shard_path = root / shard_rel
                if shard_path.is_file():
                    import zipfile
                    with zipfile.ZipFile(shard_path) as archive:
                        for file_id in shard['file_ids']:
                            yield synth.dataset.load_sample_bytes(archive.read(f"{file_id}.npz"))
                            
    synth.dataset.iter_materialized = _kaggle_auto_unzip_iter
    print("[Env] Kaggle mount detected: unzipped directory support enabled.")

from representation import V1Config
from representation.checkpoint import load_checkpoint, save_checkpoint
from representation.data import FileDataset, collate_variable_files
from representation.inference import NormalReferenceBank, RepresentationInference
from representation.model import V1RepresentationModel
from representation.trainer import RepresentationTrainer
from synth.config import PatchConfig
from synth.patchify import Patchifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Hardware] Compute device:', device)
if device.type == 'cuda':
    print('[Hardware] CUDA device name:', torch.cuda.get_device_name(0))
    print('[Hardware] Allocated memory:', f"{torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")


[Env] Loaded representation modules from: /kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01/src/src
[Env] Kaggle mount detected: unzipped directory support enabled.
[Hardware] Compute device: cuda
[Hardware] CUDA device name: Tesla T4
[Hardware] Allocated memory: 0.0 MB


In [2]:
def _resolve_default_paths() -> tuple[str, str]:
    """Auto-detect Kaggle environment vs local server environment."""
    kaggle_exact = Path('/kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01')
    if (kaggle_exact / 'manifest.json').is_file() or (kaggle_exact / 'train').is_dir():
        print(f"[Env] Detected Kaggle dataset mount: {kaggle_exact}")
        return str(kaggle_exact), '/kaggle/working/v1_representation.pt'
    
    kaggle_alt = Path('/kaggle/input/anomaly-representation-20260903-01')
    if (kaggle_alt / 'manifest.json').is_file():
        print(f"[Env] Detected Kaggle dataset mount: {kaggle_alt}")
        return str(kaggle_alt), '/kaggle/working/v1_representation.pt'
        
    if Path('/kaggle/input').is_dir():
        for p in Path('/kaggle/input').glob('**/manifest.json'):
            print(f"[Env] Discovered Kaggle dataset with manifest at: {p.parent}")
            return str(p.parent), '/kaggle/working/v1_representation.pt'
            
    local_data = os.environ.get('V1_DATA_ROOT', 'data/generated/production')
    local_ckpt = os.environ.get('V1_CHECKPOINT_PATH', 'checkpoints/v1_representation.pt')
    print(f"[Env] Detected Local/Server environment (data_root={local_data})")
    return local_data, local_ckpt

_default_data, _default_ckpt = _resolve_default_paths()

@dataclass
class TrainingParams:
    """Unified training configuration (works identically on Local Server and Kaggle GPU).
    
    Modify parameters directly here or pass keyword arguments to TrainingParams(...).
    """
    # Dataset and checkpoint paths (auto-resolved based on environment)
    data_root: str = _default_data
    checkpoint_path: str = _default_ckpt
    max_samples: int | None = int(os.environ['V1_MAX_SAMPLES']) if 'V1_MAX_SAMPLES' in os.environ else None
    resume: bool = os.environ.get('V1_RESUME', 'false').lower() in ('true', '1', 'yes')
    
    # In-memory RAM caching (recommended True for <= 25K files, False for 100K+ streaming)
    in_memory: bool = os.environ.get('V1_IN_MEMORY', 'true').lower() in ('true', '1', 'yes')
    
    # Training hyperparameters (optimized production defaults: B=128, 40 epochs)
    batch_size: int = int(os.environ.get('V1_BATCH_SIZE', '128'))
    epochs: int = int(os.environ.get('V1_EPOCHS', '40'))
    lr: float = float(os.environ.get('V1_LR', '1.5e-3'))
    weight_decay: float = float(os.environ.get('V1_WEIGHT_DECAY', '1e-4'))
    eta_min: float = float(os.environ.get('V1_ETA_MIN', '1e-5'))
    max_grad_norm: float = float(os.environ.get('V1_MAX_GRAD_NORM', '1.0'))
    
    # Contrastive schedule & loss calibration
    lambda_max: float = float(os.environ.get('V1_LAMBDA_MAX', '0.1'))  # validated user-editable parameter (conservative rerun default: 0.1)
    contrastive_warmup_steps: int = int(os.environ.get('V1_WARMUP_STEPS', str(196 * 5)))  # 5 epochs warmup
    contrastive_ramp_steps: int = int(os.environ.get('V1_RAMP_STEPS', str(196 * 5)))      # 5 epochs ramp
    contrastive_temperature: float = float(os.environ.get('V1_TEMPERATURE', '0.2'))       # calibrated tau=0.2
    
    # Model architecture
    d_model: int = int(os.environ.get('V1_D_MODEL', '128'))
    sequence_layers: int = int(os.environ.get('V1_SEQ_LAYERS', '4'))
    attention_heads: int = int(os.environ.get('V1_ATTN_HEADS', '4'))
    dropout: float = float(os.environ.get('V1_DROPOUT', '0.1'))
    
    # Selection policy
    selection_metric: str = os.environ.get('V1_SELECTION_METRIC', 'val_stationary_joint_loss')
    
    # Normal reference bank fitting
    max_ref_samples: int = int(os.environ.get('V1_MAX_REF_SAMPLES', '8192'))

    def __post_init__(self) -> None:
        if self.lambda_max < 0.0:
            raise ValueError("lambda_max must be non-negative")

# Direct instantiation
params = TrainingParams(
    epochs=40,
    lambda_max=0.1,
    contrastive_warmup_steps=196*5,
    contrastive_ramp_steps=196*5
)

configured_root = Path(params.data_root).expanduser()
repo_root = Path.cwd()
while repo_root.name in ('notebooks', 'local', 'kaggle') and not (repo_root / 'src' / 'representation').is_dir():
    repo_root = repo_root.parent
if not (repo_root / 'src' / 'representation').is_dir() and (repo_root.parent / 'src' / 'representation').is_dir():
    repo_root = repo_root.parent
DATA_ROOT = configured_root if configured_root.is_absolute() else repo_root / configured_root
MANIFEST_PATH = DATA_ROOT / 'manifest.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"V1 dataset manifest not found at {MANIFEST_PATH}. Run uv run python -m synth.cli --output {DATA_ROOT} first.")
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_splits = ('train', 'val', 'test')
splits = manifest.get('splits')
if not isinstance(splits, dict):
    raise RuntimeError(f"V1 dataset manifest at {MANIFEST_PATH} has no split mapping; regenerate with uv run python -m synth.cli.")
missing_splits = [name for name in required_splits if name not in splits]
if missing_splits:
    raise RuntimeError(f"V1 dataset at {DATA_ROOT} is missing required splits: {', '.join(missing_splits)}. Regenerate with uv run python -m synth.cli.")

def load_split(name: str, limit: int = 4):
    entry = splits[name]
    if not isinstance(entry, dict) or entry.get('status') != 'complete':
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is not complete; rerun uv run python -m synth.cli --output {DATA_ROOT} --resume.")
    samples = list(islice(FileDataset(DATA_ROOT, split=name), limit))
    if not samples:
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is empty.")
    return samples

train_samples = load_split('train')
val_samples = load_split('val')
print('dataset root', DATA_ROOT, 'manifest counts', manifest['counts'])
print('train IDs', [sample.file_id for sample in train_samples], 'val IDs', [sample.file_id for sample in val_samples])
print(f"Configured parameters: epochs={params.epochs}, batch_size={params.batch_size}, lr={params.lr}, in_memory={params.in_memory}, resume={params.resume}, lambda_max={params.lambda_max}, selection_metric={params.selection_metric}")


[Env] Detected Kaggle dataset mount: /kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01
dataset root /kaggle/input/datasets/trietp1253201581/anomaly-representation-20260903-01 manifest counts {'test': 20000, 'train': 25000, 'val': 5000}
train IDs ['N-train-51c5413776', 'N-train-89c49457e8', 'N-train-8daf1efbae', 'N-train-5d151c3e57'] val IDs ['N-val-3a33ca884c', 'N-val-a1b3618996', 'N-val-7a6966c007', 'N-val-eaafc4df9f']
Configured parameters: epochs=40, batch_size=128, lr=0.0015, in_memory=True, resume=False, lambda_max=0.1, selection_metric=val_stationary_joint_loss


In [3]:
cfg = V1Config(
    n_channels=train_samples[0].C,
    patch_size=32,
    stride=16,
    d_model=params.d_model,
    sequence_layers=params.sequence_layers,
    attention_heads=params.attention_heads,
    dropout=params.dropout,
    contrastive_weight_max=params.lambda_max,
    contrastive_warmup_steps=params.contrastive_warmup_steps,
    contrastive_ramp_steps=params.contrastive_ramp_steps,
    contrastive_temperature=params.contrastive_temperature,
)
patchifier = Patchifier(PatchConfig(patch_size=32, stride=16, pad_end=True))

class StreamingBatchDataset:
    """Yield collated minibatches, with optional Host RAM caching and full shuffling."""
    def __init__(self, data_root, split, patchifier, config, b_size=32, max_count=None, base_seed=0, in_memory=True):
        self.data_root = data_root
        self.split = split
        self.patchifier = patchifier
        self.config = config
        self.batch_size = b_size
        self.max_count = max_count
        self.base_seed = base_seed
        self.in_memory = in_memory
        self.samples = None
        
        if in_memory:
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            total = manifest['counts'][split] if self.max_count is None else min(self.max_count, manifest['counts'][split])
            self.samples = list(tqdm(iterator, total=total, desc=f"Loading {split} to RAM"))

    def __iter__(self):
        if self.samples is not None:
            indices = torch.randperm(len(self.samples)).tolist()
            chunk = []
            batch_idx = 0
            for idx in indices:
                chunk.append(self.samples[idx])
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )
        else:
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            chunk = []
            batch_idx = 0
            for sample in iterator:
                chunk.append(sample)
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )

train_batches = StreamingBatchDataset(DATA_ROOT, 'train', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=3, in_memory=params.in_memory)
val_batches = StreamingBatchDataset(DATA_ROOT, 'val', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=4, in_memory=params.in_memory)

train_batch = next(iter(train_batches))
val_batch = next(iter(val_batches))
train_total = manifest['counts']['train'] if params.max_samples is None else min(params.max_samples, manifest['counts']['train'])
val_total = manifest['counts']['val'] if params.max_samples is None else min(params.max_samples, manifest['counts']['val'])
print(f"Datasets ready: train={train_total} files, val={val_total} files, batch_size={params.batch_size}, in_memory={params.in_memory}")
print('train signals', tuple(train_batch['signals'].shape), 'validation signals', tuple(val_batch['signals'].shape), 'mask composition', train_batch['mask_composition'])


Loading train to RAM:   0%|          | 0/25000 [00:00<?, ?it/s]

Loading val to RAM:   0%|          | 0/5000 [00:00<?, ?it/s]

Datasets ready: train=25000 files, val=5000 files, batch_size=128, in_memory=True
train signals (128, 6, 794) validation signals (128, 6, 1425) mask composition [{'random': 4, 'info': 4, 'block': 4}, {'random': 4, 'info': 3, 'block': 3}, {'random': 3, 'info': 3, 'block': 3}, {'random': 5, 'info': 4, 'block': 4}, {'random': 2, 'info': 2, 'block': 2}, {'random': 3, 'info': 3, 'block': 2}, {'random': 4, 'info': 4, 'block': 4}, {'random': 2, 'info': 2, 'block': 2}, {'random': 3, 'info': 2, 'block': 2}, {'random': 4, 'info': 3, 'block': 3}, {'random': 4, 'info': 4, 'block': 3}, {'random': 3, 'info': 3, 'block': 3}, {'random': 3, 'info': 3, 'block': 2}, {'random': 4, 'info': 4, 'block': 4}, {'random': 2, 'info': 2, 'block': 2}, {'random': 5, 'info': 5, 'block': 4}, {'random': 5, 'info': 4, 'block': 4}, {'random': 2, 'info': 2, 'block': 1}, {'random': 6, 'info': 6, 'block': 6}, {'random': 6, 'info': 6, 'block': 5}, {'random': 4, 'info': 4, 'block': 4}, {'random': 4, 'info': 3, 'block': 3}, {'

In [4]:
model = V1RepresentationModel(cfg, patchifier=patchifier)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=params.lr, weight_decay=params.weight_decay)

num_batches_per_epoch = (train_total + params.batch_size - 1) // params.batch_size
num_val_batches = (val_total + params.batch_size - 1) // params.batch_size
total_steps = max(1, params.epochs * num_batches_per_epoch)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=params.eta_min)

configured_ckpt = Path(params.checkpoint_path).expanduser()
checkpoint_path = configured_ckpt if configured_ckpt.is_absolute() else repo_root / configured_ckpt
start_step = 0
if params.resume:
    if checkpoint_path.is_file():
        meta = load_checkpoint(checkpoint_path, model, optimizer=optimizer, scheduler=scheduler)
        start_step = int(meta["step"])
        print(f"[Resume] Successfully restored checkpoint from step {start_step} at {checkpoint_path} (lambda_max={params.lambda_max})")
    else:
        print(f"[Resume] Warning: Checkpoint {checkpoint_path} not found; starting fresh training.")
else:
    if checkpoint_path.is_file():
        print(f"[Training] Existing checkpoint at {checkpoint_path} will be superseded upon completion (resume=False, lambda_max={params.lambda_max}).")
    else:
        print(f"[Training] Starting fresh training (resume=False, lambda_max={params.lambda_max}).")

trainer = RepresentationTrainer(
    model,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    seed=cfg.seed,
    step=start_step,
    selection_metric=params.selection_metric,
    max_grad_norm=params.max_grad_norm,
)

print(f"Starting training: {params.epochs} epochs, {num_batches_per_epoch} batches/epoch ({total_steps} total steps, start_step={trainer.step}), lambda_max={params.lambda_max}, selection={trainer.selection_metric}, device={device}...")

history = []

epoch_pbar = tqdm(range(1, params.epochs + 1), desc="Training epochs")
for epoch in epoch_pbar:
    batch_pbar = tqdm(train_batches, total=num_batches_per_epoch, desc=f"Epoch {epoch:2d}/{params.epochs}", leave=False)
    train_metrics = trainer.train_epoch(batch_pbar)
    metrics = dict(train_metrics)
    
    val_pbar = tqdm(val_batches, total=num_val_batches, desc="Validating", leave=False)
    val_metrics = trainer.validate(val_pbar)
    metrics.update({f"val_{k}": v for k, v in val_metrics.items()})
    
    trainer.record_eval(metrics, epoch=epoch)
    
    trainer.history[-1] = metrics
    history.append(metrics)
    
    epoch_pbar.set_postfix({
        "stat_joint": f"{metrics['stationary_joint_loss']:.4f}",
        "val_stat": f"{metrics['val_stationary_joint_loss']:.4f}",
        "pred": f"{metrics['prediction_loss']:.4f}",
        "val_pred": f"{metrics['val_prediction_loss']:.4f}",
        "cont": f"{metrics['contrastive_loss']:.4f}",
        "sim": f"{metrics.get('contrastive_sim', 0.0):.3f}",
        "lambda": f"{metrics['effective_lambda']:.3f}/{params.lambda_max:.2f}",
    })

if trainer.best_state is not None:
    restored = trainer.restore_best_state()
    print(f"[Selection] Restored coherent best state from Epoch {trainer.best_epoch} (Step {trainer.step}) with score {trainer.best_loss:.4f} ({trainer.selection_metric}, lambda_max={params.lambda_max}).")
else:
    print(f"[Selection] Retaining active model state at Step {trainer.step} (selection={trainer.selection_metric}, lambda_max={params.lambda_max}).")

print('history', history)

# Fit normal reference bank on active restored model
model.eval()
max_ref = min(params.max_ref_samples, train_total)
ref_loader = StreamingBatchDataset(DATA_ROOT, 'train', patchifier, cfg, b_size=params.batch_size, max_count=max_ref, base_seed=99, in_memory=params.in_memory)
ref_total_batches = (max_ref + params.batch_size - 1) // params.batch_size
ref_embeddings_list = []
ref_pbar = tqdm(ref_loader, total=ref_total_batches, desc=f"Fitting reference bank ({max_ref} normal files)", leave=False)
with torch.no_grad():
    for batch in ref_pbar:
        dev_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        ref_out = model(dev_batch)
        ref_embeddings_list.append(ref_out['file_embedding'].cpu())
ref_embeddings = torch.cat(ref_embeddings_list, dim=0)
bank = NormalReferenceBank(k=min(cfg.knn_k, ref_embeddings.shape[0])).fit(ref_embeddings)

trainer.save_checkpoint(checkpoint_path, reference_bank=bank)
print('checkpoint', checkpoint_path, 'step', trainer.step, f"reference bank: {bank.embeddings.shape[0]} normal files")


[Training] Starting fresh training (resume=False, lambda_max=0.1).
Starting training: 40 epochs, 196 batches/epoch (7840 total steps, start_step=0), lambda_max=0.1, selection=val_stationary_joint_loss, device=cuda...


Training epochs:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch  1/40:   0%|          | 0/196 [00:00<?, ?it/s]

Validating:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch  2/40:   0%|          | 0/196 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/40 [00:00<?, ?it/s]

[Selection] Restored coherent best state from Epoch 40 (Step 7840) with score 0.1276 (val_stationary_joint_loss, lambda_max=0.1).
history [{'prediction_loss': 0.1871408529731692, 'contrastive_loss': 3.5624211187265358, 'weighted_contrastive_loss': 0.0, 'joint_loss': 0.1871408529731692, 'stationary_joint_loss': 0.5433829701980766, 'lambda': 0.0, 'effective_lambda': 0.0, 'stationary_lambda': 0.10000000000000005, 'contrastive_sim': 0.9967507069208184, 'context_norm': 11.350678390386154, 'target_norm': 11.328134619459814, 'predicted_norm': 10.728542775523906, 'file_embedding_norm': 10.433616015375877, 'step': 196.0, 'val_prediction_loss': 0.04733457639813423, 'val_contrastive_loss': 3.094423608481884, 'val_weighted_contrastive_loss': 0.0, 'val_joint_loss': 0.04733457639813423, 'val_stationary_joint_loss': 0.35677694380283353, 'val_lambda': 0.0, 'val_effective_lambda': 0.0, 'val_stationary_lambda': 0.10000000000000005, 'val_contrastive_sim': 0.9987581729888916, 'val_context_norm': 11.341538

Loading train to RAM:   0%|          | 0/8192 [00:00<?, ?it/s]

Fitting reference bank (8192 normal files):   0%|          | 0/64 [00:00<?, ?it/s]

checkpoint /kaggle/working/v1_representation.pt step 7840 reference bank: 8192 normal files


In [5]:
model.eval()
ref_batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in train_batch.items()}
with torch.no_grad():
    reference_output = model(ref_batch_device)
if bank.embeddings is None:
    bank.fit(reference_output['file_embedding'])
inference = RepresentationInference(model, bank, patchifier, masking_config=cfg)
scores = inference.score_batch(val_batch)
print('normal train reference rows', bank.embeddings.shape[0], 'step', trainer.step, 'lambda_max', params.lambda_max, 'effective_lambda', history[-1]['effective_lambda'])
print('validation S_pred', scores['S_pred'].tolist(), 'S_pop', scores['S_pop'].tolist(), 'timestep localization', scores['timestep_scores'][0].tolist())


normal train reference rows 8192 step 7840 lambda_max 0.1 effective_lambda 0.10000000000000005
validation S_pred [0.08925803750753403, 0.04429687559604645, 0.03734871372580528, 0.04258781671524048, 0.04000786319375038, 0.040513888001441956, 0.05675862357020378, 0.06574197113513947, 0.02794431522488594, 0.05221156030893326, 0.04054764285683632, 0.07029132544994354, 0.04369989410042763, 0.03983031585812569, 0.038763076066970825, 0.07332686334848404, 0.025603534653782845, 0.06645209342241287, 0.05551792308688164, 0.09996702522039413, 0.09555059671401978, 0.11210161447525024, 0.060456328094005585, 0.04686226323246956, 0.027165455743670464, 0.09486891329288483, 0.04035525023937225, 0.04503748565912247, 0.08928002417087555, 0.04250356927514076, 0.034952785819768906, 0.06195450574159622, 0.0355115607380867, 0.04703174903988838, 0.13685154914855957, 0.03906486928462982, 0.04181203618645668, 0.03201685845851898, 0.041321199387311935, 0.11769725382328033, 0.22158433496952057, 0.02898331359028816